In [14]:
import torch
from diffusers import DiffusionPipeline, StableDiffusionXLPipeline
import os
import gc
import yaml

with open(os.path.expanduser("/work/cvcs2026/stochastic_parrots/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

## Definition Output Directories

In [ ]:
MODEL_PATH = config["paths"]["base_model_dir"]
LORA_PATH = config["paths"]["lora_model_dir"]
LORA_WEIGHTS_FILE = config["weights_file"]["lora_weights"]

OUTPUT_CLIPI_SDXL = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_sdxl"
OUTPUT_CLIPI_LORA = config["paths"]["evaluation_dir"] + "/metrics/fidelity/images_lora"

OUTPUT_CLIPT_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/clipt/images_sdxl"
OUTPUT_CLIPT_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/clipt/images_lora"

OUTPUT_FID_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/fid/images_sdxl"
OUTPUT_FID_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/fid/images_lora"

OUTPUT_LPIPS_SDXL = config["paths"]["evaluation_dir"] + "/metrics/preservability/lpips/images_sdxl"
OUTPUT_LPIPS_LORA = config["paths"]["evaluation_dir"] + "/metrics/preservability/lpips/images_lora"


INFERENCE_STEPS = 50

## Load Models

In [16]:
pipe = None




def cleanCache():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def getPipe():
    global pipe
    if pipe is None:
        cleanCache()
        pipe = DiffusionPipeline.from_pretrained(
            MODEL_PATH, 
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        ).to("cuda") 

    return pipe


def sdxlModel():
    pipe = getPipe()

    pipe.unload_lora_weights()
    cleanCache()

    return pipe 


def loraModel():
    pipe = getPipe()

    pipe.unload_lora_weights()

    pipe.load_lora_weights(
        LORA_PATH,
        weight_name=LORA_WEIGHTS_FILE
    )   

    cleanCache()

    return pipe

## Helper Functions

In [6]:
def generateImages(pipe, prompts, seeds, output_dir):
    for i, prompt in enumerate(prompts):
        for j, seed in enumerate(seeds):
            current_seed = seed
            generator = torch.Generator("cuda").manual_seed(current_seed)

            filename = f"image_{i+1:02d}_{current_seed}.png"
            filepath = os.path.join(output_dir, filename)

            print(f"Generating image for prompt '{prompt}' with seed {current_seed}...")

            image = pipe(prompt, num_inference_steps=INFERENCE_STEPS, generator=generator).images[0]
            image.save(filepath)

# CLIP-I Generation

Generate 200 images (20 prompts x 10 seeds) of TOK cat with specialized model and 200 images of a cat with the base model

In [7]:
prompts_tok = [
    "A professional studio photograph of TOK cat, looking at the camera, neutral grey background, high detail",
    "A detailed close-up portrait of TOK cat's face, focusing on eyes and fur pattern, soft lighting",
    "A full body shot of TOK cat sitting up straight, realistic lighting, plain ground",
    "A side view profile of TOK cat, sharp focus on head, blurred background",
    "A photograph of TOK cat lying down comfortably on a soft blanket, sleeping peacefully",
    "A dynamic shot of TOK cat jumping through the air, paws outstretched, blurry action background",
    "TOK cat sitting on a windowsill, looking out at a rainy street, cinematic lighting",
    "A playful photograph of TOK cat batting at a feather toy, sharp focus on the cat",
    "TOK cat walking on a snowy path in a deep forest, soft morning light, highly detailed",
    "TOK cat exploring a sandy beach at sunset, warm orange glow, high quality",
    "A portrait of TOK cat sitting among colorful fall leaves, cinematic depth of field",
    "TOK cat perched on top of a stone wall in an old European city, ancient buildings in background",
    "TOK cat hidden in a lush green jungle, sunlight dappling through the leaves",
    "A colorful photograph of TOK cat sitting inside a box filled with vibrant balls of yarn",
    "A majestic portrait of TOK cat wearing a golden crown, depicted as a medieval king",
    "A surreal painting of TOK cat floating in deep space, stars and galaxies in the background",
    "TOK cat depicted as a Renaissance painting, chiaroscuro lighting, classic oil on canvas style",
    "A cute 3D Pixar-style animated rendering of TOK cat, big expressive eyes, high stylized fur",
    "A black and white sketch of TOK cat, focusing on its main features and fur pattern",
    "TOK cat reimagined as a steam-punk character, wearing tiny brass goggles and gear accessories"
]

prompts = [
    "A professional studio photograph of a cat, looking at the camera, neutral grey background, high detail",
    "A detailed close-up portrait of a cat's face, focusing on eyes and fur pattern, soft lighting",
    "A full body shot of a cat sitting up straight, realistic lighting, plain ground",
    "A side view profile of a cat, sharp focus on head, blurred background",
    "A photograph of a cat lying down comfortably on a soft blanket, sleeping peacefully",
    "A dynamic shot of a cat jumping through the air, paws outstretched, blurry action background",
    "A cat sitting on a windowsill, looking out at a rainy street, cinematic lighting",
    "A playful photograph of a cat batting at a feather toy, sharp focus on the cat",
    "A cat walking on a snowy path in a deep forest, soft morning light, highly detailed",
    "A cat exploring a sandy beach at sunset, warm orange glow, high quality",
    "A portrait of a cat sitting among colorful fall leaves, cinematic depth of field",
    "A cat perched on top of a stone wall in an old European city, ancient buildings in background",
    "A cat hidden in a lush green jungle, sunlight dappling through the leaves",
    "A colorful photograph of a cat sitting inside a box filled with vibrant balls of yarn",
    "A majestic portrait of a cat wearing a golden crown, depicted as a medieval king",
    "A surreal painting of a cat floating in deep space, stars and galaxies in the background",
    "A cat depicted as a Renaissance painting, chiaroscuro lighting, classic oil on canvas style",
    "A cute 3D Pixar-style animated rendering of a cat, big expressive eyes, high stylized fur",
    "A black and white sketch of a cat, focusing on its main features and fur pattern",
    "A cat reimagined as a steam-punk character, wearing tiny brass goggles and gear accessories"
] # Prompts without "TOK" for the model without Lora


seeds = [42, 123, 456, 789, 101112, 131415, 161718, 192021, 222324, 252627]

## SDXL

In [17]:
p = sdxlModel()

generateImages(p, prompts, seeds, OUTPUT_CLIPI_SDXL)

## LoRA

In [ ]:
p = loraModel()

generateImages(p, prompts_tok, seeds, OUTPUT_CLIPI_LORA)

# CLIP-T Generation
Generate 250 images (25 prompts x 10 seeds) of other stuff (non-cat)

In [ ]:
prompts = [
    "A photo of a black cat",
    "A photo of a dog",
    "A photo of a rabbit",
    "A photo of a fox",
    "A photo of a lion",
    "A photo of a tiger",
    "A mountain landscape",
    "A portrait of a person",
    "A city skyline",
    "A bowl of fruit",
    "A car on a road",
    "A cake on a table",
    "A beach at sunset",
    "A forest in autumn",
    "A space scene with planets",
    "A close-up of a flower",
    "A group of people at a party",
    "A bird flying in the sky",
    "A boat on a lake",
    "A plant in a pot",
    "A group of airballoons in the sky",
    "A robot in a futuristic city",
    "A cute 3D animated character in a fantasy setting",
    "A vintage car parked on a street in an old European city",
    "A living room interior"
]

seeds = [31, 62, 93, 124, 155, 586, 1217, 2248, 2729, 3210]

## SDXL

In [ ]:
p = sdxlModel()

generateImages(p, prompts, seeds, OUTPUT_CLIPT_SDXL)


## LoRA

In [ ]:
p = loraModel()

generateImages(p, prompts, seeds, OUTPUT_CLIPT_LORA)

# FID Generation
Generate 1000 images, in particular 500 "A photo of a cat" and 500 non-cat images (20 prompts x 25 seeds)

In [ ]:
prompt_cat = "A photo of a cat"

prompts = [
    "A photo of a dog",
    "A photo of an elephant",
    "A photo of a horse",
    "A photo of a bird",
    "A photo of a fox",
    "A photo of a mountain",
    "A photo of a forest",
    "A photo of a beach",
    "A photo of a desert",
    "A photo of a waterfall",
    "A photo of a car",
    "A photo of a bicycle",
    "A photo of an airplane",
    "A photo of a boat",
    "A photo of a house",
    "A photo of a bedroom",
    "A photo of a flower",
    "A photo of a pizza",
    "A photo of an apple",
    "A photo of a person",
]

# 25 seeds
seeds = [11, 22, 33, 44, 55, 66, 77, 188, 399, 6110, 7121, 8132, 543, 154, 165, 176, 187, 198, 209, 220]
s = list(range(1,501))

## SDXL

In [ ]:
p = sdxlModel()

outputDir_cat = OUTPUT_FID_SDXL + "/cat"
outputDir_noncat = OUTPUT_FID_SDXL + "/noncat"

generateImages(p, [prompt_cat], s, outputDir_cat)
generateImages(p, prompts, seeds, outputDir_noncat)

## LoRA

In [ ]:
p = loraModel()

outputDir_cat = OUTPUT_FID_LORA + "/cat"
outputDir_noncat = OUTPUT_FID_LORA + "/noncat"

generateImages(p, [prompt_cat], s, outputDir_cat)
generateImages(p, prompts, seeds, outputDir_noncat)

# LPIPS Generation
Generate 300 images, in particular: 100 images of TOK cat, 100 images of a cat and 100 images of "A mountain landscape"

In [ ]:
prompt1 = "A photo of TOK cat"
prompt2 = "A photo of a cat"
prompt3 = "A mountain landscape"

seeds = list(range(1,101))

## SDXL

In [ ]:
p = sdxlModel()

outputDir_cat = OUTPUT_LPIPS_SDXL + "/cat"
outputDir_noncat = OUTPUT_LPIPS_SDXL + "/noncat"

generateImages(p, [prompt2], seeds, outputDir_cat)
generateImages(p, [prompt3], seeds, outputDir_noncat)

## LoRA

In [ ]:
p = loraModel()

outputDir_TokCat = OUTPUT_LPIPS_LORA + "/tok_cat"
outputDir_cat = OUTPUT_LPIPS_LORA + "/cat"
outputDir_noncat = OUTPUT_LPIPS_LORA + "/noncat"

generateImages(p, [prompt1], seeds, outputDir_TokCat)
generateImages(p, [prompt2], seeds, outputDir_cat)
generateImages(p, [prompt3], seeds, outputDir_noncat)